# Data 2018 to 2021

- [Hospital Inpatient Discharges (SPARCS De-Identified)](https://health.data.ny.gov/browse?q=Hospital%20Inpatient%20Discharges%20(SPARCS%20De-Identified)&sortBy=relevance)
- [Data Access - Statewide Planning and Research Cooperative System (SPARCS)](https://www.health.ny.gov/statistics/sparcs/access/)
> - **Output Data Dictionaries: `sparcs_data_dictionary.xlsx`** SPARCS Data Dictionary - In use; use for all data extracts received October 1, 2017 and forward.
- [Hospital Inpatient Discharges (SPARCS De-Identified): 2021](https://health.data.ny.gov/Health/Hospital-Inpatient-Discharges-SPARCS-De-Identified/tg3i-cinn)
    - **Attachments: `NYSDOH_SPARCS_De-Identified_Overview_2021.pdf`**

>**Costing Methodology**
>
>Estimates of inpatient costs were calculated using hospital discharge data from SPARCS and Institutional Cost Report (ICR) data. ICR’s include data on cost for each facility as well as ratios of Cost to Charges (RCCs). RCCs are certified, calculated and reported by facilities and are subject to external audit. For example, if hospital charge is 20,000 and the RCC is 50%, the estimated cost is 10,000. As with charges, cost data are hospital-specific.
For calendar year (CY) 2021 estimated total cost for discharges was calculated using facility-specific 2019 audited RCCs file.

## 1 - Packages

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression

from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier, XGBRegressor

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

In [2]:
df = pd.read_csv('2018_2021.csv')
df = df.reindex(columns=['CCSR Procedure Description', 'Discharge Year', 'Age Group', 'Length of Stay', 
                         'Type of Admission', 'Patient Disposition', 'APR DRG Description', 
                         'APR Severity of Illness Description', 'APR Medical Surgical Description', 
                         'Payment Typology 1', 'Total Charges', 'Permanent Facility Id'])

/opt/anaconda3/envs/tf/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3457: DtypeWarning: Columns (31) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 3 - Mapping Categorical Features

In [3]:
def mapping(df):
    
    df['Length of Stay'] = df['Length of Stay'].replace({"120 +": "121"}).astype(str).astype(float) # Median: 243

    mapping0 = {"0 to 17": 1, "18 to 29": 2, "30 to 49": 3, "50 to 69": 4, "70 or Older": 5}
    df["Age Group"] = df["Age Group"].map(mapping0)
    mapping1 = {"Elective": 1, "Emergency": 2, "Newborn": 3, "Not Available": 4, "Trauma": 5, "Urgent": 6}
    df["Type of Admission"] = df["Type of Admission"].map(mapping1)
    mapping2 = {"Surgical": 1, "Medical": 2}
    df["APR Medical Surgical Description"] = df["APR Medical Surgical Description"].map(mapping2)
    
    mapping3 = {
        'Hospice - Home':1, 'Expired':2, 'Home w/ Home Health Services':3,
        'Home or Self Care':4, 'Skilled Nursing Home':5,
        'Left Against Medical Advice':6, 'Short-term Hospital':7,
        'Hospice - Medical Facility':8, 'Inpatient Rehabilitation Facility':9,
        "Cancer Center or Children's Hospital":10, 'Court/Law Enforcement':11,
        'Psychiatric Hospital or Unit of Hosp':12,
        'Medicare Cert Long Term Care Hospital':13, 'Another Type Not Listed':14,
        'Facility w/ Custodial/Supportive Care':15,
        'Federal Health Care Facility':16,
        'Hosp Basd Medicare Approved Swing Bed':17,
        'Critical Access Hospital':18, 'Medicaid Cert Nursing Facility':19
    }
    df['Patient Disposition'] = df['Patient Disposition'].map(mapping3)
    
    mapping4 = {"Minor": 1, "Moderate": 2, "Major": 3, "Extreme": 4}
    df["APR Severity of Illness Description"] = df["APR Severity of Illness Description"].map(mapping4)
    
    mapping5 = {
        'ALLOGENEIC BONE MARROW TRANSPLANT':1,
        'AUTOLOGOUS BONE MARROW TRANSPLANT OR T-CELL IMMUNOTHERAPY':2,
        'CYSTIC FIBROSIS - PULMONARY DISEASE':3,
        'EAR, NOSE, MOUTH, THROAT, CRANIAL/FACIAL MALIGNANCIES':4,
        'EXTENSIVE O.R. PROCEDURE UNRELATED TO PRINCIPAL DIAGNOSIS':5,
        'EXTENSIVE PROCEDURE UNRELATED TO PRINCIPAL DIAGNOSIS':6,
        'EXTRACORPOREAL MEMBRANE OXYGENATION (ECMO)':7,
        'MAJOR RESPIRATORY & CHEST PROCEDURES':8,
        'MAJOR RESPIRATORY AND CHEST PROCEDURES':9,
        'MODERATELY EXTENSIVE O.R. PROCEDURE UNRELATED TO PRINCIPAL DIAGNOSIS':10,
        'MODERATELY EXTENSIVE PROCEDURE UNRELATED TO PRINCIPAL DIAGNOSIS':11,
        'NON-EXTENSIVE O.R. PROCEDURE UNRELATED TO PRINCIPAL DIAGNOSIS':12,
        'NONEXTENSIVE PROCEDURE UNRELATED TO PRINCIPAL DIAGNOSIS':13,
        'OTHER RESPIRATORY & CHEST PROCEDURES':14,
        'OTHER RESPIRATORY AND CHEST PROCEDURES':15,
        'OTHER RESPIRATORY DIAGNOSES EXCEPT SIGNS, SYMPTOMS & MINOR DIAGNOSES':16,
        'OTHER RESPIRATORY DIAGNOSES EXCEPT SIGNS, SYMPTOMS AND MISCELLANEOUS DIAGNOSES':17,
        'RESPIRATORY MALIGNANCY':18,
        'RESPIRATORY SYSTEM DIAGNOSIS W VENTILATOR SUPPORT 96+ HOURS':19,
        'RESPIRATORY SYSTEM DIAGNOSIS WITH VENTILATOR SUPPORT > 96 HOURS':20,
        'TRACHEOSTOMY W MV 96+ HOURS W EXTENSIVE PROCEDURE':21,
        'TRACHEOSTOMY W MV 96+ HOURS W/O EXTENSIVE PROCEDURE':22,
        'TRACHEOSTOMY WITH MV >96 HOURS WITH EXTENSIVE PROCEDURE':23,
        'TRACHEOSTOMY WITH MV >96 HOURS WITHOUT EXTENSIVE PROCEDURE':24,
    }
    df['APR DRG Description'] = df['APR DRG Description'].map(mapping5)
    
    mapping6 = {
        'ABDOMINAL WALL PROCEDURES, NEC':1,
        'ADMINISTRATION AND TRANSFUSION OF BONE MARROW, STEM CELLS, PANCREATIC ISLET CELLS, AND T-CELLS':2,
        'ADMINISTRATION OF ALBUMIN AND GLOBULIN':3,
        'ADMINISTRATION OF ANTI-INFLAMMATORY AGENTS':4,
        'ADMINISTRATION OF ANTIBIOTICS':5,
        'ADMINISTRATION OF DIAGNOSTIC SUBSTANCES, NEC':6,
        'ADMINISTRATION OF NUTRITIONAL AND ELECTROLYTIC SUBSTANCES':7,
        'ADMINISTRATION OF THERAPEUTIC SUBSTANCES, NEC':8,
        'ADMINISTRATION OF THROMBOLYTICS AND PLATELET INHIBITORS':9,
        'ADRENALECTOMY':10,
        'AIRWAY INTUBATION':11,
        'ANEURYSM REPAIR PROCEDURES':12,
        'ANGIOPLASTY AND RELATED VESSEL PROCEDURES (ENDOVASCULAR; EXCLUDING CAROTID)':13,
        'ARTERIAL OXYGEN SATURATION MONITORING':14,
        'ARTERY, VEIN, AND GREAT VESSEL PROCEDURES, NEC':15,
        'ARTHROCENTESIS':16,
        'BEAM RADIATION':17,
        'BILIARY AND PANCREATIC CALCULUS REMOVAL':18,
        'BLADDER CATHETERIZATION AND DRAINAGE':19,
        'BONE AND JOINT BIOPSY':20,
        'BONE EXCISION':21,
        'BONE FIXATION (EXCLUDING EXTREMITIES)':22,
        'BONE MARROW BIOPSY':23,
        'BRACHYTHERAPY':24,
        'BRONCHOSCOPIC EXCISION AND FULGURATION':25,
        'BRONCHOSCOPY (DIAGNOSTIC)':26,
        'BRONCHOSCOPY (THERAPEUTIC)':27,
        'CARDIAC AND CORONARY FLUOROSCOPY':28,
        'CARDIAC CHEST COMPRESSION':29,
        'CARDIAC MONITORING':30,
        'CARDIAC STRESS TESTS':31,
        'CARDIOVASCULAR DEVICE PROCEDURES, NEC':32,
        'CARDIOVERSION':33,
        'CAROTID ENDARTERECTOMY AND STENTING':34,
        'CHEMOTHERAPY':35,
        'CHEST TUBE PLACEMENT AND THERAPEUTIC THORACENTESIS':36,
        'CHEST WALL PROCEDURES, NEC':37,
        'CLOSED REDUCTION OF BONES AND JOINTS':38,
        'CNS EXCISION PROCEDURES':39,
        'COLONOSCOPY AND PROCTOSCOPY WITH BIOPSY':40,
        'COMMON BILE DUCT SPHINCTEROTOMY AND STENTING':41,
        'COMPUTERIZED TOMOGRAPHY (CT) WITH CONTRAST':42,
        'COMPUTERIZED TOMOGRAPHY (CT) WITHOUT CONTRAST':43,
        'CONTROL OF BLEEDING (NON-ENDOSCOPIC)':44,
        'CYSTECTOMY (INCLUDING FULGURATION) AND URETHRECTOMY':45,
        'CYSTOSCOPY AND URETEROSCOPY (INCLUDING BIOPSY)':46,
        'DENTAL PROCEDURES':47,
        'DIAGNOSTIC ERCP WITH OR WITHOUT BIOPSY':48,
        'DIAPHRAGMATIC HERNIA REPAIR':49,
        'ELECTROCARDIOGRAM (ECG)':50,
        'ELECTROENCEPHALOGRAM (EEG)':51,
        'EMBOLECTOMY, ENDARTERECTOMY, AND RELATED VESSEL PROCEDURES (NON-ENDOVASCULAR; EXCLUDING CAROTID)':52,
        'ENDOCRINE SYSTEM BIOPSY':53,
        'ENDOSCOPIC CONTROL OF BLEEDING':54,
        'ENT DIAGNOSTIC ENDOSCOPY (EXCLUDING LARYNGOSCOPY)':55,
        'ENT DIAGNOSTIC PROCEDURES (NON-ENDOSCOPIC)':56,
        'ENT DRAINAGE (EXCLUDING MYRINGOTOMY)':57,
        'ENT EXCISION (EXCLUDING NASAL PASSAGE, SINUSES, TONGUE, SALIVARY GLANDS, LARYNX)':58,
        'ENT PROCEDURES, NEC':59,
        'ENT REPAIR':60,
        'ESOPHAGOGASTRODUODENOSCOPY (EGD) WITH BIOPSY':61,
        'EXPLORATION OF PERITONEAL CAVITY':62,
        'EXTRACORPOREAL MEMBRANE OXYGENATION':63,
        'EYELID PROCEDURES':64,
        'FEMALE REPRODUCTIVE SYSTEM PROCEDURES, NEC':65,
        'FEMUR FIXATION':66,
        'FIXATION OF UPPER EXTREMITY BONES':67,
        'FLUOROSCOPIC ANGIOGRAPHY (EXCLUDING CORONARY)':68,
        'FLUOROSCOPIC GUIDANCE FOR CIRCULATORY SYSTEM PROCEDURES':69,
        'FLUOROSCOPY OF NON-CIRCULATORY ORGANS':70,
        'GASTRECTOMY':71,
        'GASTRO-JEJUNAL BYPASS (INCLUDING BARIATRIC)':72,
        'GASTROSTOMY':73,
        'GI SYSTEM BIOPSY (NON-ENDOSCOPIC)':74,
        'GI SYSTEM DRAINAGE (EXCLUDING PARACENTESIS)':75,
        'GI SYSTEM ENDOSCOPIC THERAPEUTIC PROCEDURES':76,
        'GI SYSTEM ENDOSCOPY WITHOUT BIOPSY (DIAGNOSTIC)':77,
        'GI SYSTEM LYSIS OF ADHESIONS':78,
        'GI SYSTEM REPAIR (EXCLUDING ANORECTAL)':79,
        'HEART BIOPSY':80,
        'HEART CONDUCTION MECHANISM PROCEDURES':81,
        'HEMODIALYSIS':82,
        'HIP ARTHROPLASTY':83,
        'HYSTERECTOMY':84,
        'ILEOSTOMY AND COLOSTOMY':85,
        'IMMOBILIZATION BY SPLINT OR OTHER EXTERNAL DEVICE':86,
        'INCISION AND DRAINAGE OF SKIN':87,
        'INCISION AND DRAINAGE OF SUBCUTANEOUS TISSUE AND FASCIA':88,
        'INFERIOR VENA CAVA (IVC) FILTER PROCEDURES':89,
        'INFUSION OF VASOPRESSOR':90,
        'INTRACRANIAL EPIDURAL AND SUBDURAL SPACE DRAINAGE':91,
        'INTRAVENOUS INDUCTION OF LABOR':92,
        'IRRIGATION (DIAGNOSTIC AND THERAPEUTIC)':93,
        'ISOLATION PROCEDURES':94,
        'JOINT TISSUE EXCISION (EXCLUDING DISCECTOMY)':95,
        'KIDNEY AND OTHER URINARY TRACT BIOPSY (NON-ENDOSCOPIC)':96,
        'LARYNGECTOMY':97,
        'LARYNGOSCOPY (DIAGNOSTIC)':98,
        'LIGATION AND EMBOLIZATION OF VESSELS':99,
        'LIVER BIOPSY':100,
        'LOWER GI THERAPEUTIC PROCEDURES, NEC (EXCLUDING OPEN AND LAPAROSCOPIC)':101,
        'LUMBAR PUNCTURE':102,
        'LUNG, PLEURA, OR DIAPHRAGM BIOPSY (NON-ENDOSCOPIC)':103,
        'LUNG, PLEURA, OR DIAPHRAGM RESECTION (OPEN AND THORACOSCOPIC)':104,
        'LYMPH NODE BIOPSY':105,
        'LYMPH NODE DISSECTION':106,
        'LYMPH NODE EXCISION (THERAPEUTIC)':107,
        'MAGNETIC RESONANCE IMAGING (MRI)':108,
        'MASTECTOMY AND LUMPECTOMY':109,
        'MEASUREMENT AND MONITORING, NEC':110,
        'MEASUREMENT DURING CARDIAC CATHETERIZATION':111,
        'MECHANICAL VENTILATION':112,
        'MEDIASTINAL PROCEDURES, NEC':113,
        'MENTAL HEALTH PROCEDURES, NEC':114,
        'MINIMALLY INVASIVE CNS BIOPSY':115,
        'MUSCLE, TENDON, BURSA, AND LIGAMENT EXCISION':116,
        'MUSCULOSKELETAL DEVICE PROCEDURES, NEC':117,
        'NAIL PROCEDURES':118,
        'NASAL AND SINUS EXCISION':119,
        'NON-INVASIVE VENTILATION':120,
        'OPEN AND THORACOSCOPIC PLEURAL DRAINAGE':121,
        'OTHER CARDIOVASCULAR SYSTEM MEASUREMENT AND MONITORING':122,
        'OTHER GI SYSTEM DEVICE PROCEDURES':123,
        'PACEMAKER AND DEFIBRILLATOR INTERROGATION':124,
        'PACEMAKER AND DEFIBRILLATOR PROCEDURES':125,
        'PACKING AND DRESSING PROCEDURES':126,
        'PANCREATIC AND PROXIMAL BILIARY DILATION AND STENTING':127,
        'PANCREATICOBILIARY BIOPSY':128,
        'PARACENTESIS':129,
        'PERCUTANEOUS CORONARY INTERVENTIONS (PCI)':130,
        'PERICARDIAL PROCEDURES':131,
        'PERITONEAL DIALYSIS':132,
        'PHARMACOTHERAPY FOR MENTAL HEALTH (EXCLUDING SUBSTANCE USE)':133,
        'PHARMACOTHERAPY FOR SUBSTANCE USE':134,
        'PHERESIS THERAPY':135,
        'PHYSICAL, OCCUPATIONAL, AND RESPIRATORY THERAPY TREATMENT':136,
        'PLACEMENT OF TUNNELED OR IMPLANTABLE PORTION OF A VASCULAR ACCESS DEVICE':137,
        'PLAIN RADIOGRAPHY':138,
        'PLANAR NUCLEAR MEDICINE IMAGING':139,
        'POSITRON EMISSION TOMOGRAPHIC (PET) IMAGING':140,
        'POTENTIAL COVID-19 THERAPIES':141,
        'PULMONARY ARTERIAL PRESSURE MONITORING':142,
        'PULMONARY FUNCTION TESTS':143,
        'RADIATION THERAPY, NEC':144,
        'REGIONAL ANESTHESIA':145,
        'RELEASE OF LUNG AND PLEURA':146,
        'RESPIRATORY SYSTEM PROCEDURES, NEC':147,
        'RETROPERITONEAL PROCEDURES, NEC':148,
        'ROBOTIC-ASSISTED PROCEDURES':149,
        'SKIN BIOPSY AND DIAGNOSTIC DRAINAGE':150,
        'SKIN LACERATION REPAIR (EXCLUDING PERINEUM)':151,
        'SMALL BOWEL RESECTION':152,
        'SPINAL CORD DECOMPRESSION':153,
        'SPINAL EPIDURAL CATHETER PLACEMENT':154,
        'SPINE FUSION':155,
        'SUBCUTANEOUS TISSUE AND FASCIA EXCISION':156,
        'SUBCUTANEOUS TISSUE AND FASCIA PROCEDURES, NEC':157,
        'SUBCUTANEOUS TISSUE, FASCIA, AND MUSCLE BIOPSY':158,
        'SUBSTANCE USE DETOXIFICATION':159,
        'TENDON, MUSCLE, BURSA, AND LIGAMENT REPAIR (EXCLUDING PERINEAL)':160,
        'THORACENTESIS (DIAGNOSTIC)':161,
        'THYMECTOMY':162,
        'THYROIDECTOMY':163,
        'TOMOGRAPHIC NUCLEAR MEDICINE IMAGING':164,
        'TRACHEOSTOMY':165,
        'TRANSFUSION OF BLOOD AND BLOOD PRODUCTS':166,
        'TRANSFUSION OF CLOTTING FACTORS':167,
        'TRANSFUSION OF PLASMA':168,
        'ULTRASONOGRAPHY':169,
        'UPPER GI THERAPEUTIC PROCEDURES, NEC (ENDOSCOPIC)':170,
        'UPPER GI THERAPEUTIC PROCEDURES, NEC (OPEN AND LAPAROSCOPIC)':171,
        'URETER AND OTHER URINARY TRACT DILATION':172,
        'VACCINATIONS':173,
        'VENOUS AND ARTERIAL CATHETER PLACEMENT':174,
        'VESSEL REPAIR AND REPLACEMENT':175,
    }
    df['CCSR Procedure Description'] = df['CCSR Procedure Description'].map(mapping6)
    
    mapping7 = {
        'Medicare':1, 'Self-Pay':2, 'Private Health Insurance':3,
        'Blue Cross/Blue Shield':4, 'Medicaid':5, 'Federal/State/Local/VA':6,
        'Department of Corrections':7, 'Miscellaneous/Other':8,
        'Managed Care, Unspecified':9, 'Unknown':10
    }
    df['Payment Typology 1'] = df['Payment Typology 1'].map(mapping7)

    return df

#train_df = mapping(df)
train_df_ccsr = mapping(df)

In [ ]:
#train_df.head(2)
train_df_ccsr.head(2)

## 4 - Missing Value Imputation
- `CCSR Procedure Description`: **4301** cases could be consider **MCAR**, since are not related to any other values - Strategy: `Mode Imputation`
- `Permanent Facility Id`: **8** cases should be consider **MNAR**, since it states "*Redacted for Confidentiality*" - Strategy: `Classifier`
> - Check out `Discriminant Function Imputation` for further applications

In [4]:
# CCSR Procedure Description
# ct = ColumnTransformer(transformers = [('CCSR', SimpleImputer(strategy='most_frequent'), 
#                                         ['CCSR Procedure Description'])], remainder='passthrough')
# ct_CCSR = ct.fit_transform(train_df)
# train_df_ccsr = pd.DataFrame(ct_CCSR, columns = train_df.columns)

# Permanent Facility Id
def facilityId_imputation(df):
    
    train_0 = df[df['Permanent Facility Id'].notnull()] # df == train_df
    y_0 = train_0.iloc[:,11:12].values.ravel() # ValueError: Unknown label type: 'continuous' --> .astype('int')
    test_0 = df[df['Permanent Facility Id'].isnull()]   # df == train_df

    scaler = StandardScaler()
    train_0_norm = scaler.fit_transform(train_0.iloc[:,:11])
    train_0_norm = pd.DataFrame(train_0_norm, columns = train_0.iloc[:,:11].columns)
    test_0_norm = scaler.fit_transform(test_0.iloc[:,:11])
    test_0_norm = pd.DataFrame(test_0_norm, columns = test_0.iloc[:,:11].columns)
    
    #clssfr = XGBClassifier() # use_label_encoder=False --> deprecated. 
    #Streamlit - ValueError: Invalid classes inferred from unique values of `y`
    
    clssfr = Pipeline([('transform', FunctionTransformer(np.float64)), ('classifier', XGBClassifier())])
    clssfr.fit(train_0_norm, y_0)
    #clssfr = Pipeline([('transform', FunctionTransformer(np.float64)), 
                       #('classifier', LogisticRegression(max_iter=500))])
    #clssfr.fit(train_0_norm, y_0)
    
    #clssfr = LogisticRegression(max_iter=500)
    #clssfr.fit(train_0_norm, y_0)

    for i, j in enumerate(test_0.index[:len(clssfr.predict(test_0_norm))]):
        df['Permanent Facility Id'].loc[j] = clssfr.predict(test_0_norm)[i]
    
    return df

train_df_toOHE = facilityId_imputation(train_df_ccsr)

/opt/anaconda3/envs/tf/lib/python3.7/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[12:53:41] WARNING: ../src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'multi:softprob' was changed from 'merror' to 'mlogloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


/opt/anaconda3/envs/tf/lib/python3.7/site-packages/pandas/core/indexing.py:1732: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_block(indexer, value, name)


In [5]:
train_df_toOHE.isna().sum().sort_values(ascending=False)

CCSR Procedure Description             4301
Discharge Year                            0
Age Group                                 0
Length of Stay                            0
Type of Admission                         0
Patient Disposition                       0
APR DRG Description                       0
APR Severity of Illness Description       0
APR Medical Surgical Description          0
Payment Typology 1                        0
Total Charges                             0
Permanent Facility Id                     0
dtype: int64

In [6]:
train_df_toOHE

,CCSR Procedure Description,Discharge Year,Age Group,Length of Stay,Type of Admission,Patient Disposition,APR DRG Description,APR Severity of Illness Description,APR Medical Surgical Description,Payment Typology 1,Total Charges,Permanent Facility Id
0,104.0,2018,5,4.0,1,4,8,2,1,1,92851.10,1630.0
1,146.0,2018,5,1.0,1,4,14,1,1,3,34994.93,1630.0
2,105.0,2018,5,8.0,2,3,13,4,1,1,53256.78,829.0
3,104.0,2018,5,3.0,1,3,8,2,1,1,101409.19,1458.0
4,104.0,2018,4,3.0,1,4,8,3,1,1,64110.88,1740.0
...,...,...,...,...,...,...,...,...,...,...,...,...
35615,121.0,2021,4,4.0,2,3,18,2,2,4,94586.39,1463.0
35616,103.0,2021,4,4.0,2,4,15,2,1,4,120136.99,1463.0
35617,7.0,2021,5,15.0,2,4,18,4,2,1,167890.18,3058.0
35618,5.0,2021,4,24.0,2,19,18,3,2,1,269262.76,3058.0


In [7]:
train_df_toOHE.loc[35599]

CCSR Procedure Description                  NaN
Discharge Year                          2021.00
Age Group                                  5.00
Length of Stay                             1.00
Type of Admission                          2.00
Patient Disposition                        2.00
APR DRG Description                       18.00
APR Severity of Illness Description        4.00
APR Medical Surgical Description           2.00
Payment Typology 1                         1.00
Total Charges                          42455.73
Permanent Facility Id                    511.00
Name: 35599, dtype: float64

In [8]:
train_df_toOHE.loc[18989]

CCSR Procedure Description                 NaN
Discharge Year                         2019.00
Age Group                                 3.00
Length of Stay                            1.00
Type of Admission                         1.00
Patient Disposition                       2.00
APR DRG Description                      18.00
APR Severity of Illness Description       2.00
APR Medical Surgical Description          2.00
Payment Typology 1                        5.00
Total Charges                          2340.87
Permanent Facility Id                   146.00
Name: 18989, dtype: float64

# ====================================
# CCSR Procedure Description IMPUTATION

In [9]:
def ccsr_imputation(df):
    
    train_0 = df[df['CCSR Procedure Description'].notnull()] # df == train_df
    y_0 = train_0.iloc[:,:1].values.ravel() # ValueError: Unknown label type: 'continuous' --> .astype('int')
    test_0 = df[df['CCSR Procedure Description'].isnull()]   # df == train_df

    scaler = StandardScaler()
    train_0_norm = scaler.fit_transform(train_0.iloc[:,1:])
    train_0_norm = pd.DataFrame(train_0_norm, columns = train_0.iloc[:,1:].columns)
    test_0_norm = scaler.fit_transform(test_0.iloc[:,1:])
    test_0_norm = pd.DataFrame(test_0_norm, columns = test_0.iloc[:,1:].columns)
    
    #clssfr = XGBClassifier() # use_label_encoder=False --> deprecated. 
    #Streamlit - ValueError: Invalid classes inferred from unique values of `y`
    
    clssfr = Pipeline([('transform', FunctionTransformer(np.float64)), ('classifier', XGBClassifier())])
    clssfr.fit(train_0_norm, y_0)
    #clssfr = Pipeline([('transform', FunctionTransformer(np.float64)), 
                       #('classifier', LogisticRegression(max_iter=500))])
    #clssfr.fit(train_0_norm, y_0)
    
    #clssfr = LogisticRegression(max_iter=500)
    #clssfr.fit(train_0_norm, y_0)

    for i, j in enumerate(test_0.index[:len(clssfr.predict(test_0_norm))]):
        df['CCSR Procedure Description'].loc[j] = clssfr.predict(test_0_norm)[i]
    
    return df

train_df_toOHE_2 = ccsr_imputation(train_df_toOHE) #train_df_ccsr

/opt/anaconda3/envs/tf/lib/python3.7/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[13:02:31] WARNING: ../src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'multi:softprob' was changed from 'merror' to 'mlogloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


/opt/anaconda3/envs/tf/lib/python3.7/site-packages/pandas/core/indexing.py:1732: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_block(indexer, value, name)


In [10]:
train_df_toOHE_2.isna().sum().sort_values(ascending=False)

CCSR Procedure Description             0
Discharge Year                         0
Age Group                              0
Length of Stay                         0
Type of Admission                      0
Patient Disposition                    0
APR DRG Description                    0
APR Severity of Illness Description    0
APR Medical Surgical Description       0
Payment Typology 1                     0
Total Charges                          0
Permanent Facility Id                  0
dtype: int64

In [11]:
train_df_toOHE_2.loc[35599]

CCSR Procedure Description                26.00
Discharge Year                          2021.00
Age Group                                  5.00
Length of Stay                             1.00
Type of Admission                          2.00
Patient Disposition                        2.00
APR DRG Description                       18.00
APR Severity of Illness Description        4.00
APR Medical Surgical Description           2.00
Payment Typology 1                         1.00
Total Charges                          42455.73
Permanent Facility Id                    511.00
Name: 35599, dtype: float64

In [12]:
train_df_toOHE_2.loc[18989]

CCSR Procedure Description              103.00
Discharge Year                         2019.00
Age Group                                 3.00
Length of Stay                            1.00
Type of Admission                         1.00
Patient Disposition                       2.00
APR DRG Description                      18.00
APR Severity of Illness Description       2.00
APR Medical Surgical Description          2.00
Payment Typology 1                        5.00
Total Charges                          2340.87
Permanent Facility Id                   146.00
Name: 18989, dtype: float64

# ====================================

## 5 - One Hot Encoder

In [13]:
ct_oh = ColumnTransformer(transformers = [('OHE', OneHotEncoder(sparse=False), ['Age Group', 'Type of Admission', 
                                                                                'Patient Disposition', 
                                                                                'APR Medical Surgical Description',
                                                                                'Payment Typology 1'])],
                       remainder='passthrough')

ct_oh_categorical = ct_oh.fit_transform(train_df_toOHE_2) #train_df_toOHE
oh_columns = ct_oh.get_feature_names_out()
oh_columns

array(['OHE__Age Group_1', 'OHE__Age Group_2', 'OHE__Age Group_3',
       'OHE__Age Group_4', 'OHE__Age Group_5', 'OHE__Type of Admission_1',
       'OHE__Type of Admission_2', 'OHE__Type of Admission_4',
       'OHE__Type of Admission_5', 'OHE__Type of Admission_6',
       'OHE__Patient Disposition_1', 'OHE__Patient Disposition_2',
       'OHE__Patient Disposition_3', 'OHE__Patient Disposition_4',
       'OHE__Patient Disposition_5', 'OHE__Patient Disposition_6',
       'OHE__Patient Disposition_7', 'OHE__Patient Disposition_8',
       'OHE__Patient Disposition_9', 'OHE__Patient Disposition_10',
       'OHE__Patient Disposition_11', 'OHE__Patient Disposition_12',
       'OHE__Patient Disposition_13', 'OHE__Patient Disposition_14',
       'OHE__Patient Disposition_15', 'OHE__Patient Disposition_16',
       'OHE__Patient Disposition_17', 'OHE__Patient Disposition_18',
       'OHE__Patient Disposition_19',
       'OHE__APR Medical Surgical Description_1',
       'OHE__APR Medical Surgica

In [14]:
ct_oh_categorical

array([[0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 2.0000000e+00,
        9.2851100e+04, 1.6300000e+03],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 1.0000000e+00,
        3.4994930e+04, 1.6300000e+03],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 4.0000000e+00,
        5.3256780e+04, 8.2900000e+02],
       ...,
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 4.0000000e+00,
        1.6789018e+05, 3.0580000e+03],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 3.0000000e+00,
        2.6926276e+05, 3.0580000e+03],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 1.0000000e+00,
        1.1503514e+05, 1.0450000e+03]])

In [17]:
train_df = pd.DataFrame(ct_oh_categorical, columns = oh_columns)

train_df = train_df.reindex(columns=['remainder__Discharge Year', 'OHE__Age Group_1', 'OHE__Age Group_2', 
                                     'OHE__Age Group_3', 'OHE__Age Group_4', 'OHE__Age Group_5', 
                                     'OHE__Type of Admission_1',
                                       'OHE__Type of Admission_2', 'OHE__Type of Admission_4',
                                       'OHE__Type of Admission_5', 'OHE__Type of Admission_6',
                                       'OHE__Patient Disposition_1', 'OHE__Patient Disposition_2',
                                       'OHE__Patient Disposition_3', 'OHE__Patient Disposition_4',
                                       'OHE__Patient Disposition_5', 'OHE__Patient Disposition_6',
                                       'OHE__Patient Disposition_7', 'OHE__Patient Disposition_8',
                                       'OHE__Patient Disposition_9', 'OHE__Patient Disposition_10',
                                       'OHE__Patient Disposition_11', 'OHE__Patient Disposition_12',
                                       'OHE__Patient Disposition_13', 'OHE__Patient Disposition_14',
                                       'OHE__Patient Disposition_15', 'OHE__Patient Disposition_16',
                                       'OHE__Patient Disposition_17', 'OHE__Patient Disposition_18',
                                       'OHE__Patient Disposition_19',
                                       'OHE__APR Medical Surgical Description_1',
                                       'OHE__APR Medical Surgical Description_2',
                                       'OHE__Payment Typology 1_1', 'OHE__Payment Typology 1_2',
                                       'OHE__Payment Typology 1_3', 'OHE__Payment Typology 1_4',
                                       'OHE__Payment Typology 1_5', 'OHE__Payment Typology 1_6',
                                       'OHE__Payment Typology 1_7', 'OHE__Payment Typology 1_8',
                                       'OHE__Payment Typology 1_9',
                                        'remainder__CCSR Procedure Description', 'remainder__Length of Stay', 
                                        'remainder__APR DRG Description', 
                                        'remainder__APR Severity of Illness Description', 
                                        'remainder__Permanent Facility Id',
                                        'remainder__Total Charges'])

train_df.drop_duplicates(keep='first', inplace=True)

# FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. 
# `sparse_output` is ignored unless you leave `sparse` to its default value.

In [18]:
train_df

,remainder__Discharge Year,OHE__Age Group_1,OHE__Age Group_2,OHE__Age Group_3,OHE__Age Group_4,OHE__Age Group_5,OHE__Type of Admission_1,OHE__Type of Admission_2,OHE__Type of Admission_4,OHE__Type of Admission_5,OHE__Type of Admission_6,OHE__Patient Disposition_1,OHE__Patient Disposition_2,OHE__Patient Disposition_3,OHE__Patient Disposition_4,OHE__Patient Disposition_5,OHE__Patient Disposition_6,OHE__Patient Disposition_7,OHE__Patient Disposition_8,OHE__Patient Disposition_9,OHE__Patient Disposition_10,OHE__Patient Disposition_11,OHE__Patient Disposition_12,OHE__Patient Disposition_13,OHE__Patient Disposition_14,OHE__Patient Disposition_15,OHE__Patient Disposition_16,OHE__Patient Disposition_17,OHE__Patient Disposition_18,OHE__Patient Disposition_19,OHE__APR Medical Surgical Description_1,OHE__APR Medical Surgical Description_2,OHE__Payment Typology 1_1,OHE__Payment Typology 1_2,OHE__Payment Typology 1_3,OHE__Payment Typology 1_4,OHE__Payment Typology 1_5,OHE__Payment Typology 1_6,OHE__Payment Typology 1_7,OHE__Payment Typology 1_8,OHE__Payment Typology 1_9,remainder__CCSR Procedure Description,remainder__Length of Stay,remainder__APR DRG Description,remainder__APR Severity of Illness Description,remainder__Permanent Facility Id,remainder__Total Charges
0,2018.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,104.0,4.0,8.0,2.0,1630.0,92851.10
1,2018.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,146.0,1.0,14.0,1.0,1630.0,34994.93
2,2018.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,105.0,8.0,13.0,4.0,829.0,53256.78
3,2018.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,104.0,3.0,8.0,2.0,1458.0,101409.19
4,2018.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,104.0,3.0,8.0,3.0,1740.0,64110.88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35615,2021.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,121.0,4.0,18.0,2.0,1463.0,94586.39
35616,2021.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,103.0,4.0,15.0,2.0,1463.0,120136.99
35617,2021.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,15.0,18.0,4.0,3058.0,167890.18
35618,2021.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,24.0,18.0,3.0,3058.0,269262.76


## 6 - Model
We need `Discharge Year` here because data split, but not in unseen data

In [19]:
X_train = train_df.drop(train_df[train_df['remainder__Discharge Year'] > 2020].index)
X_train = X_train.iloc[:,1:-1]
X_valid = train_df.drop(train_df[train_df['remainder__Discharge Year'] < 2021].index)
X_valid = X_valid.iloc[:,1:-1]
y_train = train_df.drop(train_df[train_df['remainder__Discharge Year'] > 2020].index)
y_train = y_train.iloc[:,-1:]
y_valid = train_df.drop(train_df[train_df['remainder__Discharge Year'] < 2021].index)
y_valid = y_valid.iloc[:,-1:]

print(f"X_train Shape    : {X_train.shape}, | y_train Shape    : {y_train.shape}")
print("X_train Dimension:", X_train.ndim, "           | y_train Dimension:", y_train.ndim)
print(f"X_valid Shape    : {X_valid.shape}, | y_valid Shape    : {y_valid.shape}")
print("X_valid Dimension:", X_valid.ndim, "           | y_valid Dimension:", y_valid.ndim)

X_train Shape    : (27158, 45), | y_train Shape    : (27158, 1)
X_train Dimension: 2            | y_train Dimension: 2
X_valid Shape    : (8428, 45), | y_valid Shape    : (8428, 1)
X_valid Dimension: 2            | y_valid Dimension: 2


In [20]:
y_train_trans = np.log(y_train + 3000)
y_valid_trans = np.log(y_valid + 3000)
# model_plot_importance = XGBRegressor(n_estimators=1000, max_depth=4, learning_rate=0.1)
# model_plot_importance.fit(X_train, y_train_trans)

# https://discuss.huggingface.co/t/problem-in-xgboost-with-hosted-infernece-api/30376/10
model = Pipeline([('transform', FunctionTransformer(np.float64)), 
                  ('regressor', XGBRegressor(n_estimators=1000, max_depth=4, learning_rate=0.1))])

#model = XGBRegressor(n_estimators=1000, max_depth=4, learning_rate=0.1)
#model = LinearRegression()
model.fit(X_train, y_train_trans)

preds = model.predict(X_valid)
# Let's increase our second predictions by 7.167231% according to the Inflation Annual Change - IP (HCC 2021)
preds_ = (np.exp(preds) - 3000) * 1.0717

In [21]:
preds

array([11.517377, 10.643262, 11.556188, ..., 12.172431, 12.570829,
       11.215632], dtype=float32)

In [22]:
preds_

array([104433.016,  41699.047, 108693.06 , ..., 204034.06 , 305469.6  ,
        76393.59 ], dtype=float32)

In [23]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

print("R2_Score: ", r2_score(y_valid, preds_))
print("MSE     : ", mean_squared_error(y_valid, preds_))
print("RMSE    : ", mean_squared_error(y_valid, preds_, squared=False))
print("MAE     : ", mean_absolute_error(y_valid, preds_))
print("MAPE    : ", mean_absolute_percentage_error(y_valid, preds_))

R2_Score:  0.8446952024776083
MSE     :  2080330481.6057208
RMSE    :  45610.640004342415
MAE     :  21105.367225107737
MAPE    :  0.2062516282866858


In [ ]:
# print("R2_Score: ", r2_score(y_valid, (np.exp(preds) - 3000)))
# print("MSE     : ", mean_squared_error(y_valid, (np.exp(preds) - 3000)))
# print("RMSE    : ", mean_squared_error(y_valid, (np.exp(preds) - 3000), squared=False))
# print("MAE     : ", mean_absolute_error(y_valid, (np.exp(preds) - 3000)))
# print("MAPE    : ", mean_absolute_percentage_error(y_valid, (np.exp(preds) - 3000)))

In [ ]:
# from xgboost import plot_importance
# plot_importance(model_plot_importance, importance_type='weight');

In [ ]:
df_preds = y_valid.copy()
df_preds['Predictions'] = np.exp(preds) - 3000
df_preds['Preds_plus_Inflation'] = (np.exp(preds) - 3000) * 1.0717
df_preds

In [ ]:
df_preds.describe().apply(lambda s: s.apply('{0:.2f}'.format))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.scatterplot(data=df_preds, y='remainder__Total Charges', x=df_preds.index, color='g', marker='o', s=40, alpha=0.2,
                label='Total Charges').ticklabel_format(style='plain');

sns.scatterplot(data=df_preds, y='Preds_plus_Inflation', x=df_preds.index, color='r', marker='D', s=30, 
                label='Preds_plus_Inflation').ticklabel_format(style='plain');

## 7 - Unseen data prediction

In [ ]:
# St Barnabas Hospital == 1176
# Roswell Park Cancer Institute == 216

X = pd.DataFrame([['MEDIASTINAL PROCEDURES, NEC', 30, 2021, '50 to 69', 'Emergency', 'Home or Self Care', 
                   'RESPIRATORY MALIGNANCY', 'Moderate', 'Medical', 'Medicare', 0, 1176]],
                 
                 columns=['CCSR Procedure Description', 'Length of Stay', 'Discharge Year', 'Age Group', 
                          'Type of Admission', 'Patient Disposition', 'APR DRG Description', 
                          'APR Severity of Illness Description', 'APR Medical Surgical Description', 
                          'Payment Typology 1', 'Total Charges', 'Permanent Facility Id'])
X

In [ ]:
X = mapping(X)
X

In [ ]:
X = ct_oh.transform(X)
X = pd.DataFrame(X, columns = oh_columns)
X = X.drop(['remainder__Discharge Year', 'remainder__Total Charges'], axis=1)
X

In [ ]:
pred_X = model.predict(X)
# Let's increase our predictions by 7.167231% according to the Inflation Annual Change - IP (HCC 2021)
pred_X = (np.exp(pred_X) - 3000) * 1.0717
pred_X = pred_X.item()
print(f"The Estimated Total Charge is ${pred_X:.2f}")

## 8 - Export Model

In [ ]:
# import pickle
# pickle.dump(model,open('model.pkl', 'wb'))

In [ ]:
# import joblib
# joblib.dump(model,open('model_joblib.pkl', 'wb'))

## 9. Out of Pocket Cost

In [ ]:
def insurance():

    oop_MAX = input("How much is your Out-of-pocket Max? ")
    oop_MAX = float(oop_MAX)
    
    if pred_X < oop_MAX:

        oop_cost = 0

        deductible = input("How much is your deductible? ")
        deductible = float(deductible)

        if pred_X > deductible:
            co_insurance = input("What is your Co-Insurance (%)? ")
            co_insurance = float(co_insurance)
            co_insurance = co_insurance/100

            oop_cost = ((pred_X - deductible) * co_insurance) + deductible
            oop_cost = round(oop_cost, 2)
            oop_cost = str(oop_cost)
            print("Your estimated annual bill would be for $" + oop_cost)
        
        else:
            oop_cost = pred_X
            oop_cost = round(oop_cost, 2)
            oop_cost = str(oop_cost)
            print("Your estimated annual bill would be for $" + oop_cost)

    else:
        oop_cost = oop_MAX
        oop_cost = round(oop_cost, 2)
        oop_cost = str(oop_cost)
        print("Your estimated annual bill would be for $" + oop_cost)


def self_pay():

    oop_cost = 0

    discount = input("Facility Discount (%) / Nonprofit Support (%)? ")
    discount = float(discount)
    discount = discount/100
    oop_cost = (pred_X - (pred_X*discount))
    oop_cost = round(oop_cost, 2)
    oop_cost = str(oop_cost)
    print("Your estimated annual bill would be for $" + oop_cost)


select = """
Health Insurance or Self-Pay?

1 - Health Insurance
2 - Self-Pay

Select one option: """

option = int(input(select))

if option == 1:
    insurance()
elif option == 2:
    self_pay()
else:
    print("Please select a right option! --> 1 or 2")